03_data_cleaning.ipynb

1. Load raw combined dataset
2. Create cleaning copy
3. Standardize data types
4. Handle FL_DATE
5. Handle missing values
   5.1 Cancellation
   5.2 Delay causes
   5.3 Operational fields
   5.4 Delay fields
6. Handle duplicates
7. Handle invalid values
8. Validate logical relationships
9. Final cleaning validation
10. Save cleaned dataset
11. Cleaning summary

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path(
    r"D:\Data Analyst\EXCEL\api\processed\flights_2026_q1.csv"
)

flights = pd.read_csv(data_path)

print("Original shape:", flights.shape)

Original shape: (1847242, 43)


In [2]:
df = flights.copy()

print("Cleaning dataset shape:", df.shape)

Cleaning dataset shape: (1847242, 43)


## 1. Column Names

The BTS dataset already uses a consistent uppercase naming convention.

Therefore, column names will be retained as provided by the source.

No unnecessary renaming will be performed.

In [3]:
flights.columns.T

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'MKT_UNIQUE_CARRIER', 'BRANDED_CODE_SHARE', 'MKT_CARRIER_AIRLINE_ID',
       'MKT_CARRIER', 'MKT_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME',
       'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME',
       'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME',
       'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN',
       'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='object')

In [4]:
#view the sample data how it looks first 5.
flights.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,381.0,427.0,349.0,2475.0,0.0,0.0,43.0,0.0,0.0
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,321.0,292.0,270.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,134.0,124.0,105.0,674.0,NaN,NaN,NaN,NaN,NaN
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,229.0,210.0,191.0,1709.0,NaN,NaN,NaN,NaN,NaN
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,153.0,147.0,117.0,728.0,NaN,NaN,NaN,NaN,NaN


In [5]:
#view the sample data how it looks last 5.
flights.tail()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1847237,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,80.0,85.0,54.0,402.0,0.0,0.0,5.0,0.0,16.0
1847238,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,140.0,143.0,126.0,1020.0,NaN,NaN,NaN,NaN,NaN
1847239,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,145.0,141.0,125.0,930.0,NaN,NaN,NaN,NaN,NaN
1847240,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,95.0,102.0,85.0,395.0,NaN,NaN,NaN,NaN,NaN
1847241,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,130.0,118.0,100.0,794.0,NaN,NaN,NaN,NaN,NaN


In [6]:
#view the sample data how it looks random 5.
flights.sample()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
929921,2026,1,2,17,2,2026-02-17,DL,DL,19790,DL,...,0.0,142.0,131.0,109.0,721.0,0.0,0.0,0.0,0.0,28.0


In [7]:
#Describe is used to summerize the numeric values in table.
flights.describe()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
count,1847242.0,1847242.0,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.786458e+06,1.786268e+06,...,1.847242e+06,1.847241e+06,1.780185e+06,1.780185e+06,1.847242e+06,384426.000000,384426.000000,384426.000000,384426.000000,384426.000000
mean,2026.0,1.0,2.039273e+00,1.556970e+01,4.018039e+00,1.982750e+04,2.764304e+03,1.322166e+03,1.329818e+03,1.393807e+01,...,2.564364e-03,1.480702e+02,1.415679e+02,1.135985e+02,8.080800e+02,26.771269,5.317260,14.499233,0.130366,29.269703
std,0.0,0.0,8.309906e-01,8.701216e+00,2.017735e+00,2.667073e+02,1.724521e+03,4.851421e+02,5.011229e+02,6.111301e+01,...,5.057459e-02,7.225527e+01,7.243144e+01,7.074440e+01,5.896536e+02,82.074800,41.409899,34.940501,4.280014,64.896116
min,2026.0,1.0,1.000000e+00,1.000000e+00,1.000000e+00,1.939300e+04,1.000000e+00,1.000000e+00,1.000000e+00,-6.600000e+01,...,0.000000e+00,-8.500000e+01,1.400000e+01,6.000000e+00,3.100000e+01,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2026.0,1.0,1.000000e+00,8.000000e+00,2.000000e+00,1.979000e+04,1.332000e+03,9.080000e+02,9.130000e+02,-7.000000e+00,...,0.000000e+00,9.500000e+01,8.900000e+01,6.200000e+01,3.730000e+02,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2026.0,1.0,2.000000e+00,1.600000e+01,4.000000e+00,1.980500e+04,2.452000e+03,1.315000e+03,1.325000e+03,-2.000000e+00,...,0.000000e+00,1.310000e+02,1.250000e+02,9.700000e+01,6.570000e+02,3.000000,0.000000,0.000000,0.000000,0.000000
75%,2026.0,1.0,3.000000e+00,2.300000e+01,6.000000e+00,1.997700e+04,4.145000e+03,1.730000e+03,1.739000e+03,1.000000e+01,...,0.000000e+00,1.800000e+02,1.730000e+02,1.440000e+02,1.050000e+03,23.000000,0.000000,18.000000,0.000000,33.000000
max,2026.0,1.0,3.000000e+00,3.100000e+01,7.000000e+00,2.043600e+04,9.914000e+03,2.359000e+03,2.400000e+03,3.339000e+03,...,1.000000e+00,1.202000e+03,7.630000e+02,7.190000e+02,4.983000e+03,3339.000000,1958.000000,1560.000000,1600.000000,2338.000000


## 2. Convert Flight Date

FL_DATE represents the date on which the flight operated.


In [8]:
flights["FL_DATE"] = pd.to_datetime(
    flights["FL_DATE"],
    format="mixed",
    errors="coerce"
)

print(flights["FL_DATE"].dtype)

datetime64[ns]


In [9]:
invalid_dates = flights["FL_DATE"].isna().sum()

print("Invalid/missing flight dates:", invalid_dates)

Invalid/missing flight dates: 0


## 3. Remove Exact Duplicate Records

In [10]:
duplicate_count = flights.duplicated().sum()

print("Exact duplicates:", duplicate_count)

Exact duplicates: 0


# Handling Missing Values

In [12]:
flights.isna().sum().sort_values(ascending=False)

CANCELLATION_CODE         1784931
LATE_AIRCRAFT_DELAY       1462816
CARRIER_DELAY             1462816
SECURITY_DELAY            1462816
NAS_DELAY                 1462816
WEATHER_DELAY             1462816
AIR_TIME                    67057
ACTUAL_ELAPSED_TIME         67057
ARR_DELAY_NEW               67048
ARR_DELAY                   67048
ARR_DEL15                   67048
ARR_TIME                    62950
TAXI_IN                     62950
TAXI_OUT                    61991
DEP_DELAY                   60974
DEP_DEL15                   60974
DEP_DELAY_NEW               60974
DEP_TIME                    60784
CRS_ELAPSED_TIME                1
MONTH                           0
QUARTER                         0
YEAR                            0
BRANDED_CODE_SHARE              0
MKT_UNIQUE_CARRIER              0
FL_DATE                         0
DAY_OF_WEEK                     0
DAY_OF_MONTH                    0
MKT_CARRIER_FL_NUM              0
MKT_CARRIER                     0
MKT_CARRIER_AI

## 4. Cancellation Code

In [29]:
pd.crosstab(
    df["CANCELLED"],
    df["CANCELLATION_CODE"].isna()
)

CANCELLATION_CODE,False,True
CANCELLED,,
0.0,0,1784931
1.0,62311,0


In [30]:
cancelled_without_code = (
    (df["CANCELLED"] == 1) &
    (df["CANCELLATION_CODE"].isna())
).sum()

print(
    "Cancelled flights without cancellation code:",
    cancelled_without_code
)

Cancelled flights without cancellation code: 0


In [31]:
not_cancelled_with_code = (
    (df["CANCELLED"] == 0) &
    (df["CANCELLATION_CODE"].notna())
).sum()

print(
    "Non-cancelled flights with cancellation code:",
    not_cancelled_with_code
)

Non-cancelled flights with cancellation code: 0


## 6. Delay Values

Negative departure and arrival delays can be valid.

For example:

- DEP_DELAY = -5 means the flight departed 5 minutes early.
- ARR_DELAY = -10 means the flight arrived 10 minutes early.

Therefore, negative delay values will NOT be removed.

In [32]:
delay_cause_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

In [34]:
df[delay_cause_columns].notna().sum()

CARRIER_DELAY          384426
WEATHER_DELAY          384426
NAS_DELAY              384426
SECURITY_DELAY         384426
LATE_AIRCRAFT_DELAY    384426
dtype: int64

In [35]:
df["ARR_DELAY"].notna().sum()

np.int64(1780194)

In [36]:
df["ARR_DEL15"].value_counts(dropna=False)

ARR_DEL15
0.0    1395768
1.0     384426
NaN      67048
Name: count, dtype: int64

In [37]:
df[delay_cause_columns] = (
    df[delay_cause_columns]
    .fillna(0)
)